# Counting the Gradient Signal per Token

**Topic:** Pretraining objectives &nbsp;·&nbsp; **Difficulty:** Easy

Causal language modelling predicts every token from its predecessors, so a
sequence of length `L` yields `L - 1` prediction targets (the first token has no
predecessor). Masked language modelling
masks 15% of positions and predicts those, yielding `0.15 L`.

That factor of about seven is a real difference in how much learning signal each
forward pass produces, and it is a large part of why decoder-only pretraining
won on compute-efficiency grounds. Count it exactly, including the 80/10/10
corruption rule.

## The data (created in the next cell)

- `n_seqs = 5000` sequences of length `L = 512`.
- MLM: mask 15%, of which 80% become `[MASK]`, 10% a random token, 10% unchanged.
- Use `np.random.default_rng(55)`.
- Draw the corruption with two full `(n_seqs, L)` draws in this order:
  `sel = rng.random((n_seqs, L)) < mask_rate`, then
  `roll = rng.random((n_seqs, L))`. A selected position is masked if
  `roll < 0.8`, random if `0.8 <= roll < 0.9`, kept otherwise.

## Your task

With **NumPy only**:

1. `clm_targets` counts the number of prediction targets per sequence under causal LM
   (every position after the first).
2. Sample the MLM corruption for all sequences and count:
   `mlm_targets_mean`, the mean number of selected positions per sequence
   (count the selected positions, then divide by `n_seqs`), plus `n_masked`,
   `n_random`, `n_kept`.
3. `signal_ratio` is `clm_targets / mlm_targets_mean`.
4. `tokens_for_equal_signal` counts how many tokens MLM must process to produce as
   many prediction targets as CLM gets from `n_seqs * L` tokens.
5. Set `answers` to a dict with `"clm_targets"`, `"mlm_targets_mean"`,
   `"frac_masked"`, `"frac_random"`, `"frac_kept"`, `"signal_ratio"`,
   `"tokens_for_equal_signal"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the corruption fractions match the 80/10/10 rule and the ratio is near 7.

In [ ]:
import numpy as np

n_seqs, L = 5000, 512
mask_rate = 0.15
print(f"{n_seqs} sequences of length {L}, MLM masks {mask_rate:.0%}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("clm_targets", [], 511.0, -85.56082557020908, 1e-06),
    ("mlm_targets_mean", [], 76.6832, -12.839682777623201, 1e-06),
    ("frac_masked", [], 0.800548229599182, -0.13404220632734892, 1e-06),
    ("frac_random", [], 0.09971154046779478, -0.016695502390023154, 1e-06),
    ("frac_kept", [], 0.09974022993302314, -0.016700306097127082, 1e-06),
    ("signal_ratio", [], 6.6637803325891465, -1.1157701500486297, 1e-06),
    ("tokens_for_equal_signal", [], 17059277.651428215, -2856371.584124492, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "frac_masked" in answers:
    _r = (abs(answers["frac_masked"] - 0.8) < 0.02
          and abs(answers["frac_random"] - 0.1) < 0.02
          and abs(answers["frac_kept"] - 0.1) < 0.02)
    _ratio = 6.0 < answers["signal_ratio"] < 8.0
    print(f"  {'80/10/10 rule holds':<24} {'ok' if _r else 'wrong'}  "
          f"({answers['frac_masked']:.3f}/{answers['frac_random']:.3f}/{answers['frac_kept']:.3f})")
    print(f"  {'CLM gives ~7x signal':<24} {'ok' if _ratio else 'wrong'}  "
          f"({answers['signal_ratio']:.2f}x)")
    _ok = _ok and _r and _ratio

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The 80/10/10 split exists because a model that only ever sees `[MASK]` at
prediction time learns a representation that assumes the token is missing. The
random and unchanged cases force it to keep representing the actual token too.
Note that all three still count as prediction targets. The seven-fold signal gap
is why an encoder needs several times the tokens to match a decoder's training
signal, and it is one of the concrete reasons the field consolidated on
decoder-only models.

```python
import numpy as np

rng = np.random.default_rng(55)

clm_targets = float(L - 1)

sel = rng.random((n_seqs, L)) < mask_rate
n_sel = sel.sum()
roll = rng.random((n_seqs, L))

n_masked = float((sel & (roll < 0.8)).sum())
n_random = float((sel & (roll >= 0.8) & (roll < 0.9)).sum())
n_kept = float((sel & (roll >= 0.9)).sum())

mlm_targets_mean = float(n_sel / n_seqs)

answers = {
    "clm_targets": clm_targets,
    "mlm_targets_mean": mlm_targets_mean,
    "frac_masked": n_masked / n_sel,
    "frac_random": n_random / n_sel,
    "frac_kept": n_kept / n_sel,
    "signal_ratio": clm_targets / mlm_targets_mean,
    "tokens_for_equal_signal": float(n_seqs * L * clm_targets / mlm_targets_mean),
}
```